[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DucThinhmta/MentalHealthAI/blob/main/notebooks/evaluate_llm_colab.ipynb)

# MentalHealthAI: đo chatbot ở chế độ LLM **miễn phí** (Ollama + Qwen2.5 trên GPU Colab)

Không cần khóa API trả phí. Mô hình mở Qwen2.5 chạy ngay trên máy chủ Colab (GPU T4 miễn phí), không gửi tin nhắn ra dịch vụ bên ngoài.
Chọn `Runtime > Change runtime type > GPU (T4)`, rồi **Runtime > Run all** (khoảng 30-60 phút).

Kết quả: 100 kịch bản hội thoại (luật và LLM), bộ an toàn có thêm lớp LLM, 33 tình huống kiểm thử thăm dò. Tất cả được sao lưu vào `MyDrive/MentalHealthAI_colab/llm_eval/`.

**Ghi trung thực khi báo cáo:** đây là Qwen2.5-7B (mô hình mở, nhỏ), KHÔNG phải Claude; chất lượng trả lời và độ nhạy an toàn có thể khác nhiều. Ghi rõ tên mô hình cạnh mọi con số.

In [ ]:
import os, subprocess, sys
REPO, BRANCH = "DucThinhmta/MentalHealthAI", "main"
if not os.path.exists("/content/MentalHealthAI"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", "/content/MentalHealthAI"], check=True)
%cd /content/MentalHealthAI
sys.path.insert(0, "/content/MentalHealthAI/src")
!pip -q install -r requirements-colab.txt

## 1. Cài Ollama và tải mô hình (trên máy chủ Colab, không phải máy bạn)

In [ ]:
MODEL = "qwen2.5:7b"     # GPU T4 16 GB chạy tốt; có thể thử "qwen2.5:14b" nếu muốn chất lượng cao hơn (chậm hơn)
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, colab_helpers as ch
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
assert ch.wait_for_port(11434, timeout=60), "Ollama chưa chạy"
!ollama pull $MODEL
os.environ.update({"MHAI_LLM": "ollama", "MHAI_OLLAMA_MODEL": MODEL, "MHAI_TRANSLATOR": "none"})
import llm_client; print("LLM:", llm_client.backend(), MODEL)

In [ ]:
# Thử nhanh một câu
!python -c "import llm_client as l; print(l.generate_reply(None, 'Dạo này mình áp lực thi cử quá', []))"

## 2. Đo

In [ ]:
!python src/evaluate_dialogs.py --show-fails && cp reports/dialog_eval.json reports/dialog_eval_rules.json
!python src/evaluate_dialogs.py --llm --show-fails && cp reports/dialog_eval.json reports/dialog_eval_ollama.json && cp reports/dialog_eval.csv reports/dialog_eval_ollama.csv

In [ ]:
!python src/evaluate_safety.py --show-errors
!python src/evaluate_safety.py --llm --show-errors

In [ ]:
!python src/exploratory_test.py --llm | head -150

In [ ]:
# Tuỳ chọn: bộ chấm tự động (đồng cảm/an toàn/tự nhiên) bằng chính mô hình này. Chỉ để sàng lọc, NGƯỜI phải chấm lại.
RUN_JUDGE = False
if RUN_JUDGE:
    !python src/evaluate_dialogs.py --llm --judge && cp reports/dialog_eval.csv reports/dialog_eval_ollama_judged.csv

## 3. Lưu kết quả lên Drive (cả nhóm xem chung)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import pathlib
dst = pathlib.Path("/content/drive/MyDrive/MentalHealthAI_colab/llm_eval")
print(ch.sync_dirs([("reports", dst)], "to_drive"), "tệp đã sao lưu vào", dst)